# Da pergunta investigativa à resposta fundamentada

**Técnico em Ciência de Dados — 2º ano**  
Análise Exploratória de Dados e Inteligência de Negócios  
U3 — Projeto Análise Exploratória de Dados  
S13 — Projeto AED – Desenvolvimento das Análises  
Aula 4: Consolidação dos resultados da análise

**Objetivo:** Justificar a consistência dos resultados obtidos com base na pergunta investigativa e nas evidências produzidas.

Vocês integram a equipe de análise de dados de uma rede brasileira de oito papelarias. Ao longo do projeto, a equipe investigou diferentes condições associadas ao desempenho comercial, identificou novas necessidades de informação e descobriu oportunidades de melhoria. A direção aguarda agora a conclusão da investigação para orientar o planejamento das operações futuras.

**Pergunta investigativa:** Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?

**Missão:** Avaliem as respostas obtidas ao longo da investigação, integrem as descobertas e formulem uma resposta fundamentada à pergunta principal do projeto.

Dados simulados para fins didáticos. Tempo: 43 minutos. S6: resultados documentados em seus recortes originais. S13: 06/10 a 30/11/2025. Os recortes não são equivalentes.

## Organização do ambiente e materiais

Organizem-se em duplas, com um computador por dupla. Alternem execução e interpretação. Reúnam este notebook e os dois CSVs na mesma pasta; selecionem o kernel Python no VS Code/Jupyter e executem as células na ordem. Dependências: pandas, numpy e IPython (ambiente Jupyter). Não são necessários arquivos produzidos em outras aulas.

Arquivos: `rede_papelarias_B2_integrado.csv`, `rede_papelarias_produtos_S13A2.csv`.

## 1. Recuperem o percurso investigativo — 4 min

Na S6, a pergunta inicial foi desdobrada em três frentes:

| Frente | Pergunta intermediária | Evidência documentada na S6 |
|---|---|---|
| Atração | Quais condições estão associadas ao movimento recebido pelas lojas? | Divulgação × entradas: +0,418. |
| Conversão | Quais condições estão associadas ao aproveitamento do movimento em compras? | Espera × aproveitamento: −0,647; disponibilidade: +0,400; horas de equipe: −0,434 no conjunto da rede, +0,126 a +0,466 por loja na faixa de 150 a 199 entradas. |
| Resultado | Como os níveis de desconto das campanhas se relacionam com o aproveitamento do movimento, o lucro bruto e a margem bruta das lojas? | Desconto × aproveitamento: +0,299; margem bruta: −0,740; lucro bruto: aproximadamente zero. |

**Uma recomendação inicial possível (S6A4):** investigar alternativas para reduzir a espera. A próxima pergunta proposta naquela aula foi:

> Em quais lojas e faixas de movimento diferentes condições operacionais estão associadas a melhor aproveitamento e resultado comercial, e quais fatores ainda não registrados podem ajudar a explicar essas diferenças?

Na S7 foram selecionados, diagnosticados e preparados os dados necessários ao aprofundamento. A análise do resultado após despesas dependia do tratamento das informações de despesas.

A S13 ampliou a investigação: resultado após despesas (A1), informação por produto (A2) e significado econômico (A3). As questões a seguir retomam essa cadeia. Os resultados da S6 acima são registros das análises originais, não cálculos efetuados novamente neste notebook.

## 2. Avaliem a resposta inicial e a necessidade de aprofundamento — 9 min

Executem as células de preparação e de correlações. Aproveitamento = cupons por 100 entradas; resultado após despesas = lucro bruto menos investimento em divulgação, despesas com pessoal e outras despesas operacionais registradas. Não equivale necessariamente a lucro líquido. O código recompõe a preparação das aulas anteriores, sem preencher ausências.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

arquivos = [Path("rede_papelarias_B2_integrado.csv"), Path("rede_papelarias_produtos_S13A2.csv")]
for arquivo in arquivos:
    if not arquivo.is_file():
        raise FileNotFoundError(f"Coloquem {arquivo.name} na mesma pasta do notebook.")
original = pd.read_csv(arquivos[0], encoding="utf-8-sig")
produtos = pd.read_csv(arquivos[1], encoding="utf-8-sig", parse_dates=["data"])
# Recompor a preparação da A1, preservando ausências.
lojas = original.drop_duplicates().copy()
assert lojas.id_registro.is_unique
for coluna in ["despesas_com_pessoal", "outras_despesas_operacionais"]:
    antes = lojas[coluna]
    texto = antes.astype("string").str.strip().str.replace("R$", "", regex=False).str.strip().str.replace(",", ".", regex=False)
    convertido = pd.to_numeric(texto, errors="raise")
    assert convertido.isna().sum() == antes.isna().sum()
    lojas[coluna] = convertido.astype(float)
lojas["data"] = pd.to_datetime(lojas.data)
lojas = lojas.loc[lojas.data.between("2025-10-06", "2025-11-30")].copy()
assert lojas.entradas.gt(0).all()
lojas["cupons_por_100_entradas"] = 100 * lojas.cupons_emitidos / lojas.entradas
lojas["resultado_apos_despesas"] = (lojas.lucro_bruto - lojas.investimento_divulgacao
    - lojas.despesas_com_pessoal - lojas.outras_despesas_operacionais)

# Conferir chaves, saldos e vínculo com cada loja-dia.
produtos = produtos.sort_values(["loja", "produto", "data"]).copy()
assert produtos.id_registro_produto.is_unique
assert not produtos.duplicated(["loja", "produto", "data"]).any()
assert not produtos.isna().any().any()
assert produtos.groupby(["loja", "data"]).size().eq(6).all()
campos_un = ["estoque_inicial_un", "reposicao_un", "unidades_vendidas", "unidades_nao_atendidas", "estoque_final_un"]
assert produtos[campos_un].ge(0).all().all()
assert all(pd.api.types.is_integer_dtype(produtos[c]) for c in campos_un)
assert (produtos.estoque_inicial_un + produtos.reposicao_un - produtos.unidades_vendidas == produtos.estoque_final_un).all()
anterior = produtos.groupby(["loja", "produto"]).estoque_final_un.shift()
assert produtos.loc[anterior.notna(), "estoque_inicial_un"].eq(anterior.dropna()).all()
assert produtos.ruptura_no_dia.eq((produtos.estoque_inicial_un.eq(0) | produtos.estoque_final_un.eq(0)).astype(int)).all()
assert (produtos.unidades_nao_atendidas.eq(0) | produtos.estoque_final_un.eq(0)).all()
vinculos = produtos.merge(lojas[["id_registro", "loja", "data"]], left_on=["id_registro_loja_dia", "loja", "data"], right_on=["id_registro", "loja", "data"], how="left", validate="many_to_one")
assert vinculos.id_registro.notna().all()
# Agregar os produtos ANTES de juntar aos indicadores comerciais.
produtos["procura"] = produtos.unidades_vendidas + produtos.unidades_nao_atendidas
produtos["semana"] = (produtos.data.sub(pd.Timestamp("2025-10-06")).dt.days // 7 + 1).astype(int)
produtos["cmv_sel"] = produtos.unidades_vendidas * produtos.custo_unitario_aquisicao
produtos["abertura_zero"] = produtos.estoque_inicial_un.eq(0).astype(int)
diario = produtos.groupby(["loja", "data"], as_index=False).agg(vendidas=("unidades_vendidas", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), procura=("procura", "sum"), receita_sel=("receita_liquida_produto", "sum"), cmv_sel=("cmv_sel", "sum"), abertura_zero=("abertura_zero", "sum"))
integrado = lojas.merge(diario, on=["loja", "data"], how="left", validate="one_to_one")
assert len(integrado) == len(lojas) == len(diario)
assert integrado.procura.notna().all()
assert np.isclose(integrado.receita_liquida.sum(), lojas.receita_liquida.sum())
assert integrado.resultado_apos_despesas.isna().sum() == lojas.resultado_apos_despesas.isna().sum()
assert (integrado.receita_sel <= integrado.receita_liquida + .001).all()
assert (integrado.cmv_sel <= integrado.receita_liquida - integrado.lucro_bruto + .001).all()
assert (integrado.receita_sel - integrado.cmv_sel <= integrado.lucro_bruto + .001).all()
assert (integrado.receita_sel >= integrado.cmv_sel - .001).all()
assert integrado.loc[integrado.disponibilidade_produtos_pct.eq(100), "abertura_zero"].eq(0).all()

print("Loja-dias:",len(integrado),"| Registros de produtos:",len(produtos))
# Proporção pela razão entre totais; denominador zero permanece indefinido.
chave = ["loja", "produto"]
por_produto = produtos.groupby(chave).agg(descricao=("descricao_produto", "first"), procura=("procura", "sum"), vendas=("unidades_vendidas", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), estoque_mediano=("estoque_final_un", "median"))
por_produto["pct_nao_atendida"] = 100 * por_produto.nao_atendidas / por_produto.procura.replace(0, np.nan)
semanal = produtos.groupby(chave + ["semana"]).agg(procura=("procura", "sum"), nao_atendidas=("unidades_nao_atendidas", "sum"), estoque_fim=("estoque_final_un", "last"))
por_produto["semanas_com_falta"] = semanal.nao_atendidas.gt(0).groupby(level=[0,1]).sum()
ordem = por_produto.sort_values("nao_atendidas", ascending=False)

# Triagem exploratória: referência de duas semanas de vendas médias.
por_produto["referencia_un"] = np.ceil(2 * por_produto.vendas / 8).astype(int)
semanal_ref = semanal.join(por_produto.referencia_un)
por_produto["semanas_acima"] = semanal_ref.estoque_fim.gt(semanal_ref.referencia_un).groupby(level=[0,1]).sum()
por_produto["candidato"] = por_produto.estoque_mediano.gt(por_produto.referencia_un) & por_produto.semanas_acima.ge(6)
candidatos = por_produto.loc[por_produto.candidato].copy()
rotulos = {"descricao":"Produto", "vendas":"Vendas em 8 semanas (un.)", "referencia_un":"Referência (un.)", "estoque_mediano":"Estoque final mediano (un.)", "semanas_acima":"Fechamentos acima da referência (de 8)"}

# Procurar faltas e estoques do mesmo produto em outras lojas na mesma data.
com_ref = produtos.join(por_produto[["referencia_un", "candidato"]], on=["loja", "produto"])
falta = com_ref.loc[com_ref.unidades_nao_atendidas.gt(0), ["data", "produto", "loja", "unidades_nao_atendidas"]].rename(columns={"loja":"loja_falta"})
estoque = com_ref.loc[com_ref.candidato & com_ref.estoque_inicial_un.gt(com_ref.referencia_un) & com_ref.estoque_final_un.gt(com_ref.referencia_un) & com_ref.ruptura_no_dia.eq(0), ["data", "produto", "loja"]].rename(columns={"loja":"loja_estoque"})
coincidencias = falta.merge(estoque, on=["data", "produto"]).query("loja_falta != loja_estoque")
pares = coincidencias.groupby(["produto", "loja_falta", "loja_estoque"]).agg(dias_coincidentes=("data", "nunique"), unidades_nas_datas=("unidades_nao_atendidas", "sum")).sort_values("dias_coincidentes", ascending=False)

print("Contrastes de estoque e procura não atendida — oito semanas")
display(pares.head(2).rename(columns={"dias_coincidentes":"Dias coincidentes", "unidades_nas_datas":"Unidades não atendidas"}))
print("Critério de triagem: mediana acima de duas semanas de vendas médias e pelo menos seis fechamentos semanais acima da referência; nas datas coincidentes, saldo de abertura e fechamento acima da referência, sem ruptura.")


Loja-dias: 384 | Registros de produtos: 2304
Contrastes de estoque e procura não atendida — oito semanas


,,,Dias coincidentes,Unidades não atendidas
produto,loja_falta,loja_estoque,,
P01,L3,L4,23,292
P03,L5,L2,19,117


Critério de triagem: mediana acima de duas semanas de vendas médias e pelo menos seis fechamentos semanais acima da referência; nas datas coincidentes, saldo de abertura e fechamento acima da referência, sem ruptura.


<details>
<summary>Dicionário dos dados complementares (consulta, se necessário)</summary>

### Dicionário dos dados complementares

| Campo | Unidade/tipo | Definição |
|---|---|---|
| id_registro_produto | texto | Identificador único do registro de produto. |
| id_registro_loja_dia | texto | Identificador do registro da loja e dia no CSV integrado da A1. |
| data | AAAA-MM-DD | Data da operação da loja. |
| loja | texto | Identificador da loja, L1 a L8. |
| produto | texto | Identificador do produto, P01 a P06. |
| descricao_produto | texto | Denominação comercial simulada. |
| estoque_inicial_un | inteiro, unidades | Estoque na abertura, antes de reposições do dia. |
| reposicao_un | inteiro, unidades | Quantidade recebida após a abertura e antes das vendas nesse modelo. Não inclui transferências entre lojas. |
| unidades_vendidas | inteiro, unidades | Unidades do produto vendidas no dia. Não equivale a número de cupons. |
| unidades_nao_atendidas | inteiro, unidades | Quantidade solicitada e não fornecida por indisponibilidade do produto (registro de solicitações, não clientes distintos nem vendas perdidas). |
| ruptura_no_dia | inteiro (0/1) | 1 se o estoque esteve zerado na abertura ou atingiu zero após as vendas; 0 caso contrário. Pode haver ruptura sem solicitação registrada não atendida. |
| estoque_final_un | inteiro, unidades | Saldo no fechamento do dia. |
| custo_unitario_aquisicao | R$/unidade | Custo simulado de aquisição unitária, constante por produto. |
| receita_liquida_produto | R$ | Receita do produto no dia, descontada a campanha simulada. |
| volume_unitario_l | litros/unidade | Volume externo aproximado por unidade embalada, assumido na simulação; não inclui corredores, prateleiras ou folgas. |

A disponibilidade geral refere-se à abertura da loja. Os seis produtos não representam todo o sortimento nem toda a procura. O resultado após despesas registradas não equivale a lucro líquido; cupons por 100 entradas não identifica pessoas distintas.

</details>

Os registros por produto detalham seis produtos, não todo o sortimento. Procura registrada = unidades vendidas + unidades não atendidas. A proporção agregada é a razão entre as somas, não a média de proporções. Os coeficientes são de Pearson. A comparação entre lojas não é pareada por data. As diferenças entre os coeficientes da S6 e da S13 não representam necessariamente mudanças temporais.

In [2]:
# Preservar o conjunto de casos completos da A1 e conferir o denominador adicional.
integrado["pct_nao_atendida"] = 100 * integrado.nao_atendidas / integrado.procura.replace(0, np.nan)
condicoes = ["espera_media_min", "horas_equipe", "disponibilidade_produtos_pct", "pct_nao_atendida"]
desempenho = ["cupons_por_100_entradas", "receita_liquida", "resultado_apos_despesas"]
variaveis_a1 = ["espera_media_min", "horas_equipe", "disponibilidade_produtos_pct"] + desempenho
completos_a1 = integrado.dropna(subset=variaveis_a1).copy()
completos = completos_a1.dropna(subset=["pct_nao_atendida"]).copy()
faixa = completos.loc[completos.entradas.between(150, 199)].copy()
print("Loja-dias no período:", len(integrado))
print("Casos completos da A1:", len(completos_a1))
print("Casos completos comuns:", len(completos))
print("Casos completos comuns na faixa de 150 a 199 entradas:", len(faixa))
rotulos_corr = {"espera_media_min":"Espera média", "horas_equipe":"Horas de equipe", "disponibilidade_produtos_pct":"Disponibilidade geral", "pct_nao_atendida":"Procura não atendida (%)", "cupons_por_100_entradas":"Cupons por 100 entradas", "receita_liquida":"Receita líquida", "resultado_apos_despesas":"Resultado após despesas"}
def correlacoes(tab):
    return tab[condicoes + desempenho].corr(method="pearson").loc[condicoes, desempenho]
corr_periodo, corr_faixa = correlacoes(completos), correlacoes(faixa)
print("Pearson — período completo")
display(corr_periodo.rename(index=rotulos_corr, columns=rotulos_corr).round(3))
print("Pearson — faixa de movimento")
display(corr_faixa.rename(index=rotulos_corr, columns=rotulos_corr).round(3))
# Comparar as oito lojas; proporção agregada pela razão dos totais.
def resumo_lojas(tab):
    res = tab.groupby("loja").agg(n=("id_registro", "size"), disponibilidade=("disponibilidade_produtos_pct", "mean"), espera=("espera_media_min", "mean"), aproveitamento=("cupons_por_100_entradas", "mean"), receita=("receita_liquida", "mean"), resultado=("resultado_apos_despesas", "mean"), procura=("procura", "sum"), nao_atendidas=("nao_atendidas", "sum"))
    res["pct_nao_atendida"] = 100 * res.nao_atendidas / res.procura.replace(0, np.nan)
    return res
lojas_periodo, lojas_faixa = resumo_lojas(completos), resumo_lojas(faixa)
rotulos = {"n":"Observações", "disponibilidade":"Disponibilidade geral (%)", "espera":"Espera média (min)", "nao_atendidas":"Não atendida (un.)", "procura":"Procura (un.)", "pct_nao_atendida":"Não atendida (%)", "aproveitamento":"Cupons por 100 entradas", "receita":"Receita média (R$)", "resultado":"Resultado médio (R$)"}
print("Quatro lojas — faixa de 150 a 199 entradas")
display(lojas_faixa.loc[["L3","L4","L5","L8"], ["n","disponibilidade","pct_nao_atendida","aproveitamento","receita","resultado"]].rename(columns=rotulos).round(2))

Loja-dias no período: 384
Casos completos da A1: 363
Casos completos comuns: 363
Casos completos comuns na faixa de 150 a 199 entradas: 125
Pearson — período completo


,Cupons por 100 entradas,Receita líquida,Resultado após despesas
Espera média,-0.649,-0.001,-0.233
Horas de equipe,-0.451,0.337,-0.249
Disponibilidade geral,0.346,0.262,0.393
Procura não atendida (%),-0.222,0.171,-0.046


Pearson — faixa de movimento


,Cupons por 100 entradas,Receita líquida,Resultado após despesas
Espera média,-0.344,-0.314,-0.099
Horas de equipe,0.047,0.266,-0.159
Disponibilidade geral,0.282,0.315,0.325
Procura não atendida (%),-0.080,-0.015,0.017


Quatro lojas — faixa de 150 a 199 entradas


,Observações,Disponibilidade geral (%),Não atendida (%),Cupons por 100 entradas,Receita média (R$),Resultado médio (R$)
loja,,,,,,
L3,14,96.80,9.09,23.28,3182.36,-50.02
L4,8,95.56,2.83,36.23,4142.40,317.65
L5,8,95.89,2.77,17.92,2486.05,-136.15
L8,14,95.36,5.67,32.06,3561.75,200.75


**Q1.** Avaliem a contribuição das três frentes investigadas na S6 para responder à pergunta principal. Considerando também as descobertas da S13A1, justifiquem por que a recomendação inicial de investigar a redução da espera precisou ser reavaliada.

Registrem as respostas avaliadas e a contribuição de cada uma para o próximo passo. Usem frases curtas e evidências.

| Pergunta investigada | Resposta avaliada (com evidência) | Contribuição para a continuidade |
|---|---|---|
| Atração | | |
| Conversão | | |
| Resultado | | |
| Relações com resultado após despesas | | |

**Por que reavaliar a recomendação inicial?**



## 3. Avaliem a consistência das novas descobertas — 10 min

Retomem a lacuna da A1: a disponibilidade geral e a espera não esclareciam suficientemente as diferenças entre lojas. Comparem L3 e L4 na faixa de 150 a 199 entradas. A informação por produto permitiu investigar a procura não atendida e os contrastes dos mesmos produtos em lojas diferentes.

Verifiquem a recorrência desses contrastes nas primeiras e nas últimas quatro semanas. Mantemos a mesma referência exploratória da A2, calculada nas oito semanas: duas semanas de vendas médias, arredondadas para cima. Não é estoque ideal. Cada intervalo contém 24 dias de operação por loja.

In [3]:
display(lojas_faixa.loc[["L3","L4"], ["n","espera","disponibilidade","pct_nao_atendida","resultado"]].rename(columns=rotulos).round(2))
linhas_rec = []
for cod, destino, origem in [("P01","L3","L4"),("P03","L5","L2")]:
    for inicio,fim,rotulo in [(1,4,"Primeiras 4 semanas"),(5,8,"Últimas 4 semanas")]:
        dest = produtos.loc[produtos.produto.eq(cod) & produtos.loja.eq(destino) & produtos.semana.between(inicio,fim)]
        c = coincidencias.loc[coincidencias.produto.eq(cod) & coincidencias.loja_falta.eq(destino) & coincidencias.loja_estoque.eq(origem)].copy()
        semanas_c = (c.data.sub(pd.Timestamp("2025-10-06")).dt.days // 7 + 1)
        sem = semanal_ref.loc[(origem,cod)].loc[inicio:fim]
        assert len(dest)==24 and len(sem)==4
        linhas_rec.append({"Produto":dest.descricao_produto.iloc[0],"Loja com falta":destino,"Loja com estoque elevado":origem,"Intervalo":rotulo,"Unidades não atendidas":int(dest.unidades_nao_atendidas.sum()),"Dias coincidentes":int(c.loc[semanas_c.between(inicio,fim),"data"].nunique()),"Fechamentos acima da referência (de 4)":int(sem.estoque_fim.gt(sem.referencia_un).sum())})
recorrencia = pd.DataFrame(linhas_rec)
display(recorrencia)

,Observações,Espera média (min),Disponibilidade geral (%),Não atendida (%),Resultado médio (R$)
loja,,,,,
L3,14,9.02,96.80,9.09,-50.02
L4,8,8.89,95.56,2.83,317.65


,Produto,Loja com falta,Loja com estoque elevado,Intervalo,Unidades não atendidas,Dias coincidentes,Fechamentos acima da referência (de 4)
0,Caderno universitário,L3,L4,Primeiras 4 semanas,191,14,4
1,Caderno universitário,L3,L4,Últimas 4 semanas,124,9,3
2,Resma de papel A4 (500 folhas),L5,L2,Primeiras 4 semanas,62,8,3
3,Resma de papel A4 (500 folhas),L5,L2,Últimas 4 semanas,61,11,4


**Q2.** Avaliem se as informações complementares obtidas por produto justificaram o aprofundamento da investigação. Considerando também os resultados dos dois intervalos, julguem se existem evidências suficientes para considerar os desequilíbrios identificados um problema recorrente no período investigado.

| Pergunta investigada | Resposta avaliada (com evidência) | Contribuição para a continuidade |
|---|---|---|
| Contribuição das informações por produto e recorrência dos contrastes | | |



## 4. Avaliem a relevância estratégica das descobertas — 8 min

Recuperem o cenário retrospectivo da A3 (24 a 29/11/2025). As duas alternativas atendem à mesma procura adicional: comprar mais mercadorias ou aproveitar os estoques existentes. Mantêm-se as vendas e reposições registradas e pressupõe-se disponibilidade no momento necessário. O cenário não é uma intervenção realizada nem uma ordem de transferência atual. O código reproduz os resultados da A3; não é necessário refazer os cálculos manualmente.

In [4]:
# Cenário retrospectivo: mesmos dias, vendas e reposições observados.
# O aproveitamento de estoque é representado por uma alocação no início da semana.
semana_final = produtos.loc[produtos.data.between("2025-11-24", "2025-11-29")].copy()
casos = [("P01", "L3", "L4"), ("P03", "L5", "L2")]
linhas = []
for cod, destino, origem in casos:
    d = semana_final.query("produto == @cod and loja == @destino").sort_values("data")
    o = semana_final.query("produto == @cod and loja == @origem").sort_values("data")
    h = produtos.query("produto == @cod and loja == @destino")
    assert len(d) == len(o) == 6
    n = int(d.unidades_nao_atendidas.sum())
    custo = h.custo_unitario_aquisicao.unique(); volume = h.volume_unitario_l.unique()
    assert len(custo) == len(volume) == 1
    custo, volume = float(custo[0]), float(volume[0])
    assert o.custo_unitario_aquisicao.eq(custo).all() and o.volume_unitario_l.eq(volume).all()
    preco = h.receita_liquida_produto.sum() / h.unidades_vendidas.sum()
    # Saldo de origem após destinar n unidades; saldo de destino após atender a procura.
    origem_abertura = o.estoque_inicial_un - n
    origem_fechamento = o.estoque_final_un - n
    destino_fechamento = d.estoque_final_un + n - d.unidades_nao_atendidas.cumsum()
    destino_abertura = d.estoque_inicial_un + n - d.unidades_nao_atendidas.cumsum().shift(fill_value=0)
    assert min(origem_abertura.min(), origem_fechamento.min()) > 0
    assert min(destino_abertura.min(), destino_fechamento.min()) >= 0
    saldo_compra = int(o.estoque_final_un.iloc[-1] + destino_fechamento.iloc[-1])
    saldo_aproveita = int(origem_fechamento.iloc[-1] + destino_fechamento.iloc[-1])
    linhas.append({"Produto": h.descricao_produto.iloc[0], "Loja com falta": destino, "Loja com estoque": origem,
        "Unidades adicionais": n, "Compras adicionais (R$)": n*custo,
        "Estoque final — compra (un.)": saldo_compra, "Estoque final — aproveitamento (un.)": saldo_aproveita,
        "Receita adicional (R$)": n*preco, "Lucro bruto adicional (R$)": n*(preco-custo), "Volume embalado a menos (L)": n*volume})
cenario = pd.DataFrame(linhas).set_index("Produto")
print("Procura registrada não atendida — 24 a 29/11/2025")
display(cenario[["Loja com falta", "Loja com estoque", "Unidades adicionais"]])
print("Comparação das alternativas (mesmo atendimento adicional)")
comparacao = pd.DataFrame({"Compra adicional": [cenario["Unidades adicionais"].sum(), cenario["Compras adicionais (R$)"].sum(), cenario["Estoque final — compra (un.)"].sum()],
    "Aproveitamento de estoques existentes": [0, 0, cenario["Estoque final — aproveitamento (un.)"].sum()]}, index=["Compras adicionais (un.)", "Desembolso em compras adicionais (R$)", "Estoque final (un.)"])
display(comparacao.round(2))


Procura registrada não atendida — 24 a 29/11/2025


,Loja com falta,Loja com estoque,Unidades adicionais
Produto,,,
Caderno universitário,L3,L4,13
Resma de papel A4 (500 folhas),L5,L2,19


Comparação das alternativas (mesmo atendimento adicional)


,Compra adicional,Aproveitamento de estoques existentes
Compras adicionais (un.),32.0,0
Desembolso em compras adicionais (R$),564.5,0
Estoque final (un.),65.0,33


In [5]:
# As duas alternativas atendem à mesma procura no cenário.
cols = ["Receita adicional (R$)", "Lucro bruto adicional (R$)", "Volume embalado a menos (L)"]
economia = cenario[cols].copy()
economia.loc["Total"] = economia.sum()
display(economia.round(2))
print(f"Diferença de desembolso em compras: R$ {cenario['Compras adicionais (R$)'].sum():.2f}")


,Receita adicional (R$),Lucro bruto adicional (R$),Volume embalado a menos (L)
Produto,,,
Caderno universitário,301.73,121.03,8.45
Resma de papel A4 (500 folhas),535.56,151.76,60.80
Total,837.29,272.79,69.25


Diferença de desembolso em compras: R$ 564.50


**Q3.** Considerando as descobertas de toda a investigação, avaliem a importância de aperfeiçoar o planejamento do abastecimento em relação à recomendação inicial de investigar a redução da espera. Justifiquem sua avaliação considerando as evidências e os benefícios econômicos identificados.

Considerem oportunidades de venda com margem positiva e custos de manutenção de estoques desnecessários. O menor desembolso em compras não é lucro; não somem os dois valores.

| Pergunta investigada | Resposta avaliada (com evidência) | Contribuição para a continuidade |
|---|---|---|
| Significado econômico das descobertas | | |



## 5. Construam e revisem a resposta final — 12 min

> **Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?**

**Q4.** Formulem uma resposta completa à pergunta principal do projeto. Integrem as respostas às perguntas intermediárias e as evidências produzidas durante a investigação. Justifiquem a prioridade recomendada, explicando como as descobertas posteriores contribuíram para rever ou sustentar a recomendação inicial.

**Condições associadas ao desempenho comercial:**


**Prioridade de melhoria e sua justificativa:**


**Alcance da conclusão:**


Releiam os registros anteriores e verifiquem se responderam às duas partes da pergunta inicial, se a recomendação decorre das evidências e se distinguem resultados observados de cenários hipotéticos.

## Critérios para avaliação da atividade

- Relaciona as perguntas intermediárias à pergunta inicial.
- Justifica a consistência das descobertas com evidências.
- Integra resultados da S6, S7 e S13.
- Responde às duas partes da pergunta principal e defende uma prioridade fundamentada.
- Distingue resultados observados de estimativas hipotéticas e delimita o alcance da conclusão.

Confiram as respostas com o professor e salvem o notebook preenchido. Este é o registro da atividade; não é necessário transcrever as respostas para o roteiro. A modalidade de envio será orientada pelo professor.

# Expectativa de respostas — uso exclusivo do professor

## Q1

Atração: divulgação × entradas, r = +0,418; maior investimento associado a mais movimento, sem medida de retorno. Conversão: espera × aproveitamento, −0,647; disponibilidade × aproveitamento, +0,400; horas de equipe × aproveitamento, −0,434 no agregado, mas +0,126 a +0,466 por loja na faixa de 150 a 199 entradas. Resultado: desconto × aproveitamento, +0,299; margem bruta, −0,740; lucro bruto, aproximadamente zero. Há um trade-off entre aproveitamento e margem, sem desconto ótimo demonstrado. A S7 selecionou, diagnosticou e preparou os dados; as despesas exigiam tratamento. Na S13A1, espera × resultado após despesas: −0,233 e −0,099; disponibilidade × resultado: +0,393 e +0,325 (período e faixa). A espera continua relevante; a prioridade inicial era defensável, mas a inclusão do resultado após despesas e as diferenças entre lojas justificam reavaliá-la. Correlações não medem o efeito de uma intervenção.

## Q2

Na faixa de 150 a 199 entradas, L3 tem disponibilidade geral 96,80%, procura não atendida de 9,09% e resultado médio −R$ 50,02; L4 tem 95,56%, 2,83% e R$ 317,65. O agregado de disponibilidade não identifica os produtos procurados. Nas primeiras e últimas quatro semanas, respectivamente: cadernos L3, 191 e 124 unidades não atendidas; L3/L4, 14 e 9 dias coincidentes; L4, 4 e 3 fechamentos semanais acima da referência. Resmas L5, 62 e 61 unidades não atendidas; L5/L2, 8 e 11 dias coincidentes; L2, 3 e 4 fechamentos acima da referência. Os contrastes aparecem nos dois intervalos, com intensidades diferentes. A triagem não determina estoque ideal e a recorrência dentro da amostra não garante persistência futura. As associações globais da procura não atendida com o resultado são fracas (−0,046 e +0,017); não generalizar a explicação do desempenho da rede.

## Q3

Na última semana, o cenário permite atender 32 unidades adicionais (13 cadernos e 19 resmas), com receita adicional de R$ 837,29 e lucro bruto adicional de R$ 272,79. Para o mesmo atendimento, o aproveitamento dos estoques exige R$ 564,50 a menos em compras e termina com 33 unidades, ante 65 da alternativa de compra; 69,25 L a menos de volume embalado. O cenário ilustra duas oportunidades: vendas com margem positiva e menor necessidade de compras e de manutenção de estoques. Desembolso evitado não é lucro e os valores não se somam; não há economia de manutenção quantificada. O planejamento futuro pode ser aperfeiçoado independentemente da viabilidade de remanejamentos atuais. Não se conhece o retorno comparável de todas as alternativas.

## Q4

Condições associadas: divulgação e movimento; espera e aproveitamento; disponibilidade e indicadores comerciais; horas de equipe com relações dependentes do recorte e do indicador; descontos com maior aproveitamento e menor margem. O resultado após despesas amplia a leitura do desempenho. As informações por produto revelam procura não atendida e desequilíbrios recorrentes localizados. Uma prioridade bem sustentada é aperfeiçoar compras, reposições e distribuição por loja e produto, considerando os padrões de procura. A cadeia de evidências liga a revisão da prioridade inicial, a lacuna informacional, os resultados por produto, a recorrência e o cenário econômico. Há oportunidades de vendas com margem positiva e de reduzir estoques desnecessários e custos de manutenção. O cenário é hipotético, os dados são simulados, a associação não é causal e não se comprovou maior retorno entre todas as alternativas. Aceitar outras prioridades justificadas com evidências e limites; não exigir a recomendação de referência.